In [ ]:
# Single end-to-end script: training, evaluation, fairness metrics, SHAP explainability

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, classification_report, roc_curve, auc,
    precision_recall_curve, confusion_matrix
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# -------------------------
# 1. Load and clean data
# -------------------------
DATA_PATH = r"C:\Projects_AiML\pro ethics\compas\cox-violent-parsed.csv"
df = pd.read_csv(DATA_PATH)

# Drop columns not used in analysis (keeps script consistent with your original)
columns_to_drop = [
    'id','first','last','in_custody','vr_charge_desc','r_case_number','vr_charge_degree',
    'c_offense_date','c_case_number','r_offense_date','juv_other_count','end','event',
    'screening_date','start','juv_misd_count','juv_fel_count','r_days_from_arrest',
    'r_charge_degree','days_b_screening_arrest','vr_case_number','priors_count.1','r_jail_out',
    'c_arrest_date','r_charge_desc','r_jail_in','violent_recid','decile_score.1',
    'vr_offense_date','out_custody'
]
df.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')

# Keep rows with required columns present
required_cols = ['decile_score','priors_count','race','sex','age_cat','is_violent_recid','v_decile_score','is_recid']
df = df.dropna(subset=required_cols).copy()

# Target: replace -1 with 0 if present
df['is_recid'] = df['is_recid'].replace(-1, 0).astype(int)

# -------------------------
# 2. Feature selection
# -------------------------
FEATURES = ['decile_score','priors_count','race','sex','age_cat','is_violent_recid','v_decile_score']
TARGET = 'is_recid'

X = df[FEATURES].copy()
y = df[TARGET].copy()

# -------------------------
# 3. Train/test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 4. Preprocessing pipeline
# -------------------------
categorical_cols = ['race', 'sex', 'age_cat']
numeric_cols = ['decile_score', 'priors_count', 'v_decile_score', 'is_violent_recid']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='drop'
)

# -------------------------
# 5. Model and hyperparameter tuning
# -------------------------
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42, use_label_encoder=False)

pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', xgb)])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.01, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

# -------------------------
# 6. Predictions and evaluation
# -------------------------
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("\nOverall accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("\nConfusion matrix:\n", cm)
print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")

# ROC and PR curves
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC (AUC={roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR (AUC={pr_auc:.3f})', color='green', linewidth=2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# -------------------------
# 7. Predicted recidivism rates and class prevalence
# -------------------------
pred_rate_overall = y_pred.mean()
true_rate_overall = y_test.mean()
print(f"\nPredicted recidivism rate (test): {pred_rate_overall:.4f}")
print(f"True recidivism rate (test): {true_rate_overall:.4f}")

# -------------------------
# 8. Fairness analysis by group
# -------------------------
def group_fairness_table(X_test, y_test, y_pred, y_pred_proba, group_col):
    df_eval = X_test.copy()
    df_eval = df_eval.reset_index(drop=True)
    df_eval['y_true'] = y_test.reset_index(drop=True)
    df_eval['y_pred'] = y_pred
    df_eval['y_score'] = y_pred_proba

    groups = df_eval[group_col].unique()
    rows = []
    for g in sorted(groups, key=lambda x: str(x)):
        sub = df_eval[df_eval[group_col] == g]
        n = len(sub)
        if n == 0:
            continue
        pred_rate = sub['y_pred'].mean()
        true_rate = sub['y_true'].mean()
        cm = confusion_matrix(sub['y_true'], sub['y_pred'], labels=[0,1])
        if cm.shape == (2,2):
            tn, fp, fn, tp = cm.ravel()
        else:
            tn = fp = fn = tp = 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        rows.append({
            group_col: g,
            'n': n,
            'pred_rate': pred_rate,
            'true_rate': true_rate,
            'fpr': fpr,
            'fnr': fnr,
            'tpr (recall)': tpr,
            'precision': precision
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

# Evaluate by race, sex, age_cat
for col in ['race', 'sex', 'age_cat']:
    print(f"\nGroup fairness metrics by {col}:")
    table = group_fairness_table(X_test, y_test, y_pred, y_pred_proba, col)
    # Add parity gaps relative to overall predicted rate
    table['pred_rate_gap'] = table['pred_rate'] - pred_rate_overall
    display_cols = ['n', 'pred_rate', 'true_rate', 'pred_rate_gap', 'fpr', 'fnr', 'tpr (recall)', 'precision']
    print(table[display_cols].to_string(index=False))

# Visualize predicted rates by group
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
sns.barplot(x='race', y='pred_rate', data=group_fairness_table(X_test, y_test, y_pred, y_pred_proba, 'race'))
plt.title('Predicted recidivism rate by race'); plt.xticks(rotation=45)

plt.subplot(1, 3, 2)
sns.barplot(x='sex', y='pred_rate', data=group_fairness_table(X_test, y_test, y_pred, y_pred_proba, 'sex'))
plt.title('Predicted recidivism rate by sex')

plt.subplot(1, 3, 3)
sns.barplot(x='age_cat', y='pred_rate', data=group_fairness_table(X_test, y_test, y_pred, y_pred_proba, 'age_cat'))
plt.title('Predicted recidivism rate by age_cat'); plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

# -------------------------
# 9. Disparity metrics (differences)
# -------------------------
def disparity_report(df_table, baseline=None, metric='pred_rate'):
    if baseline is None:
        baseline = df_table[metric].mean()
    df_table['gap_vs_baseline'] = df_table[metric] - baseline
    df_table['ratio_vs_baseline'] = df_table[metric] / baseline
    return df_table

race_table = group_fairness_table(X_test, y_test, y_pred, y_pred_proba, 'race')
race_disp = disparity_report(race_table.copy(), baseline=pred_rate_overall, metric='pred_rate')
print("\nRace disparity (predicted rate gaps):")
print(race_disp[['race','n','pred_rate','gap_vs_baseline','ratio_vs_baseline']].to_string(index=False))

# -------------------------
# 10. Calibration check (optional but useful)
# -------------------------
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Calibration curve')
plt.plot([0,1],[0,1],'k--', label='Perfectly calibrated')
plt.xlabel('Mean predicted probability'); plt.ylabel('Fraction of positives')
plt.title('Calibration plot'); plt.legend(); plt.grid(alpha=0.3); plt.show()

# -------------------------
# 11. SHAP explainability
# -------------------------
# Extract the trained classifier and the preprocessor
trained_clf = best_model.named_steps['classifier']
preproc = best_model.named_steps['preprocessor']

# Transform a sample of training data for SHAP (use a subset for speed)
X_train_trans = preproc.transform(X_train)
X_test_trans = preproc.transform(X_test)

# Get feature names after preprocessing
try:
    feature_names = preproc.get_feature_names_out()
except Exception:
    # Fallback: build names manually
    num_names = numeric_cols
    cat_encoder = preproc.named_transformers_['cat']
    cat_names = list(cat_encoder.get_feature_names_out(categorical_cols))
    feature_names = np.array(list(num_names) + cat_names)

# Use TreeExplainer for XGBoost
explainer = shap.TreeExplainer(trained_clf)
# Use a small background sample for speed
background = X_train_trans[np.random.choice(X_train_trans.shape[0], min(1000, X_train_trans.shape[0]), replace=False)]
shap_values = explainer.shap_values(background)

# Global summary (bar + beeswarm)
# For binary TreeExplainer shap_values is a 1D or 2D array depending on shap version; handle both
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, background, feature_names=feature_names, show=True)
plt.tight_layout()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, background, feature_names=feature_names, plot_type="bar", show=True)
plt.tight_layout()

# -------------------------
# 12. Local explanation: force plot for one test instance
# -------------------------
# Choose an index (e.g., first positive prediction or a specific index)
# Prefer an instance where model predicted recidivism (y_pred == 1)
test_df = X_test.reset_index(drop=True).copy()
test_preds = pd.Series(y_pred, index=test_df.index)
positive_indices = test_preds[test_preds == 1].index
if len(positive_indices) > 0:
    idx = positive_indices[0]
else:
    idx = 0

# Transform the single instance
x_instance_trans = preproc.transform(test_df.loc[[idx]])
# Compute SHAP values for the instance
shap_vals_instance = explainer.shap_values(x_instance_trans)

# Force plot (in notebook this will render inline; in script we save a matplotlib figure)
# Use shap.plots.force for interactive; here we create a static bar-like explanation
shap_df = pd.DataFrame({
    'feature': feature_names,
    'shap_value': shap_vals_instance.flatten() if shap_vals_instance.ndim == 2 else shap_vals_instance
})
shap_df['abs_shap'] = np.abs(shap_df['shap_value'])
shap_df = shap_df.sort_values('abs_shap', ascending=False).head(15)

plt.figure(figsize=(8, 6))
colors = ['red' if v > 0 else 'blue' for v in shap_df['shap_value']]
plt.barh(shap_df['feature'][::-1], shap_df['shap_value'][::-1], color=colors[::-1])
plt.xlabel('SHAP value (impact on log-odds/probability)')
plt.title(f'Local explanation (top 15) for test index {idx}')
plt.grid(alpha=0.2, axis='x')
plt.tight_layout(); plt.show()

# Print local details
print("\nLocal instance details (original features):")
print(test_df.loc[idx])
print("\nModel predicted probability:", y_pred_proba[idx])
print("Model predicted class:", y_pred[idx])
print("\nTop SHAP contributions (absolute):")
print(shap_df[['feature','shap_value']].to_string(index=False))

# -------------------------
# 13. Save key figures (optional)
# -------------------------
# Create output folder
out_dir = "compas_xgb_outputs"
os.makedirs(out_dir, exist_ok=True)
# Example: save ROC and PR plots if desired (already shown inline)
# plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=300)

# -------------------------
# 14. Detailed interpretation guide
# -------------------------
print("\n" + "="*80)
print("INTERPRETATION GUIDE - UNDERSTANDING BIAS IN THE MODEL")
print("="*80)

print("\n📊 OVERALL MODEL PERFORMANCE:")
print("-" * 80)
print(f"The model achieves {accuracy_score(y_test, y_pred):.1%} accuracy overall.")
print(f"However, accuracy alone doesn't tell us if the model is FAIR across different groups.")
print(f"We need to examine how it performs for different races, genders, and age groups.")

print("\n🔍 WHAT THE FAIRNESS METRICS MEAN:")
print("-" * 80)
print("• PREDICTED RATE: What % the model predicts will re-offend in each group")
print("  → If much higher for one race, the model may be BIASED against that group")
print(f"  → Overall predicted rate: {pred_rate_overall:.1%}")
print()
print("• FPR (False Positive Rate): % of people who WON'T re-offend but are predicted to")
print("  → High FPR for a group = model UNFAIRLY labels innocent people in that group")
print()
print("• FNR (False Negative Rate): % of people who WILL re-offend but are predicted safe")
print("  → High FNR for a group = model MISSES dangerous people in that group")
print()
print("• TPR (True Positive Rate/Recall): % of actual re-offenders correctly identified")
print("  → Low TPR = model fails to catch re-offenders in that group")

print("\n⚠️ SIGNS OF BIAS TO LOOK FOR:")
print("-" * 80)
print("1. PREDICTED RATE GAPS: If one racial group has much higher predicted rates")
print("   → Example: 50% for African-Americans vs 30% for Caucasians = BIAS")
print()
print("2. FPR IMBALANCE: If innocent people in one group are mislabeled more often")
print("   → This means the model is UNFAIRLY harsh on that group")
print()
print("3. FNR IMBALANCE: If dangerous people in one group are missed more often")
print("   → This means the model gives that group MORE BENEFIT OF DOUBT")
print()
print("4. You CANNOT have equal FPR AND FNR across groups if base rates differ")
print("   → This is a fundamental fairness tradeoff - you must choose which to prioritize")

print("\n🎯 GLOBAL SHAP ANALYSIS (What drives predictions overall?):")
print("-" * 80)
print("Look at the SHAP bar chart (blue bars) - features at the top matter MOST:")
print()
print("• num__priors_count & num__is_violent_recid at top = LEGITIMATE predictors")
print("  → These SHOULD influence predictions (criminal history matters)")
print()
print("• cat__race_XXX or cat__sex_XXX near the top = POTENTIAL BIAS")
print("  → Race/gender SHOULD NOT be strong predictors in a fair system")
print("  → If they rank high, the model is using protected attributes unfairly")
print()
print("• Look at the beeswarm plot (colorful dots):")
print("  → Red dots (high feature value) pointing RIGHT = increases recidivism prediction")
print("  → Blue dots (low feature value) pointing LEFT = decreases recidivism prediction")

print("\n👤 LOCAL SHAP EXPLANATION (Why did the model predict THIS person?):")
print("-" * 80)
print("The bar chart for test index shows which features pushed THIS prediction:")
print()
print("• RED bars pointing RIGHT = features that INCREASED recidivism risk for this person")
print("• BLUE bars pointing LEFT = features that DECREASED risk")
print()
print("If you see strong influence from race/gender features:")
print("  → This person's prediction was BIASED by their demographic attributes")
print("  → Even if prior count is high, race shouldn't add extra punishment")

print("\n📈 CALIBRATION PLOT (Are predicted probabilities accurate?):")
print("-" * 80)
print("The calibration plot shows if a 60% predicted probability really means 60% risk:")
print()
print("• Points ON the diagonal line = perfectly calibrated (honest probabilities)")
print("• Points ABOVE line = model is underconfident (predicts 60% but actually 80% risk)")
print("• Points BELOW line = model is overconfident (predicts 60% but actually 40% risk)")
print()
print("Check if calibration differs by race - if so, predictions are LESS TRUSTWORTHY")
print("for certain groups even if overall accuracy seems good.")

print("\n⚖️ FAIRNESS RECOMMENDATIONS:")
print("-" * 80)
print("Based on your results, look for these RED FLAGS:")
print()
print(f"1. Check if African-Americans have predicted rate > {pred_rate_overall + 0.1:.1%}")
print("   (more than 10 percentage points above average) → DEMOGRAPHIC BIAS")
print()
print("2. Check if FPR for any race is >10 percentage points higher than others")
print("   → That group faces UNFAIR false accusations")
print()
print("3. Check if race features rank in top 5 of SHAP importance")
print("   → Model is DIRECTLY using race to make decisions (likely illegal)")
print()
print("4. In local explanations, if race adds >0.3 SHAP value for ANY individual")
print("   → That person's score was SIGNIFICANTLY inflated by their race")

print("\n✅ WHAT TO DO IF YOU FIND BIAS:")
print("-" * 80)
print("• REMOVE race/gender from features entirely (try training without them)")
print("• Use FAIRNESS CONSTRAINTS during training (e.g., equalized odds)")
print("• Examine if priors_count itself is biased (reflects over-policing)")
print("• Consider if the OUTCOME (is_recid) has measurement bias")
print("• Implement POST-PROCESSING to equalize FPR or FNR across groups")
print("• Most importantly: DOCUMENT the bias and decide if the system should be used at all")

print("\n" + "="*80)
print("END OF INTERPRETATION GUIDE")
print("="*80 + "\n")